Import the relevant Python libraries

In [1]:
import sqlite3
import csv
import os

Mount the Google drive where the SQL source file is stored. This allows the code to access the data stored on Google Drive.


Find the file path in Google drive for the data file

In [2]:
# Original file path with spaces
original_file_path = "data/student-scores.csv"

Check that the file exists in the expected location

In [3]:
# Check if the file exists
if not os.path.exists(original_file_path):
    print(f"Error: CSV file not found at '{original_file_path}'")
else:
    print(f"CSV file found at '{original_file_path}'")

CSV file found at 'data/student-scores.csv'


Create the table object in SQLite. If the table already exists drop it - this stops the same data being added twice to the table

In [4]:
# Connect to SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('student_scores.db')
cursor = conn.cursor()

# Drop the table if it exists - this stops the table having duplicate rows added when the program runs
cursor.execute('DROP TABLE IF EXISTS my_table')

Read the CSV file. Use the header of the file to generate the table headers. This will add all of the data to the table. As no data types are specified everything is added as text.

In [5]:
# Open the CSV file and read headers
with open(original_file_path, 'r') as file:
    reader = csv.reader(file)
    headers = next(reader)  # Get the header row

    # Create a table using the headers
    columns = ', '.join([f'{header} TEXT' for header in headers])
    cursor.execute(f'CREATE TABLE IF NOT EXISTS my_table ({columns})')

    # Insert data into the table
    for row in reader:
        placeholders = ', '.join(['?' for _ in headers])
        cursor.execute(f'INSERT INTO my_table ({", ".join(headers)}) VALUES ({placeholders})', row)

# Commit the transaction and close the connection
conn.commit()

Print out the number of rows in the table - with the sample data should be 2000

In [6]:
# Print the number of rows in the table
cursor.execute('SELECT COUNT(*) FROM my_table')
row_count = cursor.fetchone()[0]
print(f'Number of rows in the table: {row_count}')

Number of rows in the table: 2000


Method to print the data returns by the SQL command. Adds the row number and strips out the parentheses and commas

In [7]:
def run_sql_command(cursor, command_str):
    # Query and print the table contents
    cursor.execute(command_str)
    rows = cursor.fetchall()
    i = 0
    for row in rows:
        # Print the row number and the row data - remove the brackets from the tuple
        print(str(i) + ":" + str(row).strip("()").replace("'", "").replace(",", ""))
        i+=1

SQL to print out all of the contents of a table

In [10]:
print("All students:")
run_sql_command(cursor, """SELECT * FROM my_table""")

All students:
0:1 Paul Casey paul.casey.1@gslingacademy.com male False 3 False 27 Lawyer 73 81 93 97 63 80 87
1:2 Danielle Sandoval danielle.sandoval.2@gslingacademy.com female False 2 False 47 Doctor 90 86 96 100 90 88 90
2:3 Tina Andrews tina.andrews.3@gslingacademy.com female False 9 True 13 Government Officer 81 97 95 96 65 77 94
3:4 Tara Clark tara.clark.4@gslingacademy.com female False 5 False 3 Artist 71 74 88 80 89 63 86
4:5 Anthony Campos anthony.campos.5@gslingacademy.com male False 5 False 10 Unknown 84 77 65 65 80 74 76
5:6 Kelly Wade kelly.wade.6@gslingacademy.com female False 2 False 26 Unknown 93 100 67 78 72 80 84
6:7 Anthony Smith anthony.smith.7@gslingacademy.com male False 3 True 23 Software Engineer 99 96 97 73 88 76 64
7:8 George Short george.short.8@gslingacademy.com male True 2 True 34 Software Engineer 95 95 82 63 84 70 85
8:9 Stanley Gutierrez stanley.gutierrez.9@gslingacademy.com male False 6 False 25 Unknown 94 68 94 85 81 74 72
9:10 Audrey Simpson audrey.sim

SQL to print out the contents of two columns

In [11]:
print("All students with their last name and email:")
run_sql_command(cursor, """SELECT last_name, email
  FROM my_table""")

All students with their last name and email:
0:Casey paul.casey.1@gslingacademy.com
1:Sandoval danielle.sandoval.2@gslingacademy.com
2:Andrews tina.andrews.3@gslingacademy.com
3:Clark tara.clark.4@gslingacademy.com
4:Campos anthony.campos.5@gslingacademy.com
5:Wade kelly.wade.6@gslingacademy.com
6:Smith anthony.smith.7@gslingacademy.com
7:Short george.short.8@gslingacademy.com
8:Gutierrez stanley.gutierrez.9@gslingacademy.com
9:Simpson audrey.simpson.10@gslingacademy.com
10:White gabrielle.white.11@gslingacademy.com
11:Randolph clinton.randolph.12@gslingacademy.com
12:Gomez patricia.gomez.13@gslingacademy.com
13:Jackson pamela.jackson.14@gslingacademy.com
14:Jackson laura.jackson.15@gslingacademy.com
15:Wiley roger.wiley.16@gslingacademy.com
16:Thompson vicki.thompson.17@gslingacademy.com
17:Davidson maxwell.davidson.18@gslingacademy.com
18:Werner jonathan.werner.19@gslingacademy.com
19:Rios angela.rios.20@gslingacademy.com
20:Nichols tim.nichols.21@gslingacademy.com
21:Willis kyle.wil

SQL to print all rows which match a specific criteria in a field

In [12]:
print("All students who have a part-time job:")
run_sql_command(cursor, """SELECT * FROM my_table
  WHERE part_time_job = 'True'""")

All students who have a part-time job:
0:8 George Short george.short.8@gslingacademy.com male True 2 True 34 Software Engineer 95 95 82 63 84 70 85
1:13 Patricia Gomez patricia.gomez.13@gslingacademy.com female True 7 False 4 Business Owner 94 59 69 67 89 65 73
2:21 Tim Nichols tim.nichols.21@gslingacademy.com male True 3 False 15 Software Engineer 100 90 72 98 73 97 72
3:42 Jesus Rasmussen jesus.rasmussen.42@gslingacademy.com male True 7 False 4 Business Owner 65 55 66 50 88 56 86
4:43 Lauren Farmer lauren.farmer.43@gslingacademy.com female True 6 False 3 Business Owner 93 94 92 77 87 98 63
5:45 David Gillespie david.gillespie.45@gslingacademy.com male True 4 False 3 Business Owner 54 99 79 57 53 95 83
6:46 Kenneth Davis kenneth.davis.46@gslingacademy.com male True 5 True 5 Business Owner 88 53 50 58 84 53 96
7:58 David Vaughn david.vaughn.58@gslingacademy.com male True 2 False 18 Banker 94 78 73 97 67 78 99
8:66 Kristy Weber kristy.weber.66@gslingacademy.com female True 6 False 2 Bus

SQL to print out the number of items in a table matcing a specific criteria

In [13]:
print("The number of students who have a part-time job:")
run_sql_command(cursor, """SELECT COUNT(*) FROM my_table
  WHERE part_time_job = 'True'""")

The number of students who have a part-time job:
0:316


SQL to print out the contents of a table sorted in ascending order by a specific field (last_name)

In [14]:
print("All students sorted by last name (ascending):")
run_sql_command(cursor, """SELECT * FROM my_table
  ORDER BY last_name ASC""")

All students sorted by last name (ascending):
0:1277 Hayley Abbott hayley.abbott.1277@gslingacademy.com female False 0 False 1 Game Developer 91 66 91 62 82 69 62
1:736 Beth Acosta beth.acosta.736@gslingacademy.com female False 7 False 15 Designer 80 91 100 87 87 94 100
2:1064 James Acosta james.acosta.1064@gslingacademy.com male True 1 False 35 Scientist 85 90 98 92 89 88 95
3:235 Linda Adams linda.adams.235@gslingacademy.com female False 1 False 20 Software Engineer 96 67 88 95 82 75 84
4:351 Aimee Adams aimee.adams.351@gslingacademy.com female False 3 False 15 Teacher 76 64 64 60 75 92 61
5:375 Kristen Adams kristen.adams.375@gslingacademy.com female False 5 True 21 Unknown 77 72 81 63 69 73 80
6:862 Brian Adams brian.adams.862@gslingacademy.com male False 5 False 34 Scientist 92 81 84 84 84 66 74
7:915 Heidi Adams heidi.adams.915@gslingacademy.com female True 10 False 3 Business Owner 43 83 58 89 87 78 86
8:1086 Nathaniel Adams nathaniel.adams.1086@gslingacademy.com male False 2 Fa

SQL to print out the contents of a table sorted in descending order by a speific field (last_name)

In [15]:
print("All students sorted by last name (descending):")
run_sql_command(cursor, """SELECT * FROM my_table
  ORDER BY last_name DESC""")

All students sorted by last name (descending):
0:287 Penny Zuniga penny.zuniga.287@gslingacademy.com female False 7 True 30 Lawyer 84 97 60 100 83 88 94
1:1671 Charlotte Zuniga charlotte.zuniga.1671@gslingacademy.com female False 8 False 34 Lawyer 83 94 93 62 92 98 94
2:1233 Toni Zimmerman toni.zimmerman.1233@gslingacademy.com female False 5 False 31 Accountant 94 98 74 94 35 95 80
3:1101 Sabrina Zavala sabrina.zavala.1101@gslingacademy.com female False 1 False 31 Scientist 92 78 96 91 82 98 100
4:484 Julie Zamora julie.zamora.484@gslingacademy.com female False 6 True 1 Unknown 76 82 93 98 98 60 65
5:1542 Stephanie Yu stephanie.yu.1542@gslingacademy.com female False 4 False 2 Unknown 71 68 77 73 68 73 64
6:147 Mark Young mark.young.147@gslingacademy.com male False 8 False 2 Business Owner 68 97 51 93 68 58 78
7:340 James Young james.young.340@gslingacademy.com male False 9 False 4 Business Owner 45 98 95 66 84 83 100
8:399 Denise Young denise.young.399@gslingacademy.com female False 2 

SQL to print out the distinct fields in a table matcing a specific criteria - useful for seeing the number of unique items in a column

In [16]:
print("All unique career aspirations:")
run_sql_command(cursor, """SELECT DISTINCT career_aspiration
  FROM my_table""")

All unique career aspirations:
0:Lawyer
1:Doctor
2:Government Officer
3:Artist
4:Unknown
5:Software Engineer
6:Teacher
7:Business Owner
8:Scientist
9:Banker
10:Writer
11:Accountant
12:Designer
13:Construction Engineer
14:Game Developer
15:Stock Investor
16:Real Estate Developer


SQL to view all data in a row where the physics_score is > 90. As the data is stored as a String it needs to be cast to an integer.

In [17]:
print("All physics grades greater than 90%:")
run_sql_command(cursor, """SELECT * FROM my_table
  WHERE CAST(physics_score AS INTEGER) > 90""")

All physics grades greater than 90%:
0:1 Paul Casey paul.casey.1@gslingacademy.com male False 3 False 27 Lawyer 73 81 93 97 63 80 87
1:2 Danielle Sandoval danielle.sandoval.2@gslingacademy.com female False 2 False 47 Doctor 90 86 96 100 90 88 90
2:3 Tina Andrews tina.andrews.3@gslingacademy.com female False 9 True 13 Government Officer 81 97 95 96 65 77 94
3:7 Anthony Smith anthony.smith.7@gslingacademy.com male False 3 True 23 Software Engineer 99 96 97 73 88 76 64
4:9 Stanley Gutierrez stanley.gutierrez.9@gslingacademy.com male False 6 False 25 Unknown 94 68 94 85 81 74 72
5:11 Gabrielle White gabrielle.white.11@gslingacademy.com female False 2 False 7 Teacher 65 60 97 94 71 81 66
6:12 Clinton Randolph clinton.randolph.12@gslingacademy.com male False 1 False 7 Unknown 80 61 100 65 87 64 61
7:17 Vicki Thompson vicki.thompson.17@gslingacademy.com female False 3 True 30 Scientist 92 64 93 91 80 89 72
8:19 Jonathan Werner jonathan.werner.19@gslingacademy.com male False 1 False 37 Doctor 

SQL to print out the distribution of items in a field

In [18]:
print("Distribution of math scores:")
run_sql_command(cursor, """SELECT math_score FROM my_table""")

Distribution of math scores:
0:73
1:90
2:81
3:71
4:84
5:93
6:99
7:95
8:94
9:98
10:65
11:80
12:94
13:66
14:96
15:94
16:92
17:86
18:92
19:99
20:100
21:57
22:89
23:50
24:87
25:92
26:100
27:64
28:79
29:82
30:88
31:70
32:77
33:99
34:90
35:65
36:99
37:92
38:77
39:92
40:85
41:65
42:93
43:97
44:54
45:88
46:71
47:81
48:80
49:100
50:76
51:99
52:64
53:90
54:85
55:91
56:95
57:94
58:94
59:89
60:91
61:83
62:98
63:83
64:91
65:90
66:92
67:64
68:90
69:99
70:89
71:62
72:61
73:51
74:85
75:99
76:87
77:75
78:88
79:87
80:87
81:76
82:60
83:85
84:73
85:85
86:91
87:99
88:93
89:89
90:95
91:68
92:98
93:79
94:78
95:100
96:86
97:87
98:100
99:85
100:87
101:96
102:76
103:90
104:65
105:98
106:91
107:86
108:91
109:78
110:76
111:82
112:45
113:92
114:43
115:96
116:78
117:68
118:93
119:55
120:66
121:98
122:94
123:96
124:69
125:77
126:89
127:96
128:82
129:99
130:96
131:95
132:90
133:88
134:87
135:62
136:85
137:79
138:90
139:97
140:88
141:71
142:70
143:93
144:92
145:83
146:68
147:92
148:69
149:76
150:86
151:41
152:91
153:7

SQL to print out all items between a range. Data is cast as an integer in this case

In [19]:
print("All math grades between 95% and 100%:")
run_sql_command(cursor, """SELECT * FROM my_table
  WHERE CAST(math_score AS INTEGER) BETWEEN 95 AND 100""")

All math grades between 95% and 100%:
0:7 Anthony Smith anthony.smith.7@gslingacademy.com male False 3 True 23 Software Engineer 99 96 97 73 88 76 64
1:8 George Short george.short.8@gslingacademy.com male True 2 True 34 Software Engineer 95 95 82 63 84 70 85
2:10 Audrey Simpson audrey.simpson.10@gslingacademy.com female False 3 True 18 Teacher 98 69 88 71 67 71 73
3:15 Laura Jackson laura.jackson.15@gslingacademy.com female False 3 False 39 Doctor 96 90 86 92 92 95 87
4:20 Angela Rios angela.rios.20@gslingacademy.com female False 2 False 27 Software Engineer 99 65 98 75 66 72 100
5:21 Tim Nichols tim.nichols.21@gslingacademy.com male True 3 False 15 Software Engineer 100 90 72 98 73 97 72
6:27 Jason Williams jason.williams.27@gslingacademy.com male False 3 False 34 Banker 100 77 80 94 63 90 90
7:34 Phyllis Diaz phyllis.diaz.34@gslingacademy.com female False 1 False 24 Banker 99 84 84 78 78 77 68
8:37 Ryan Lee ryan.lee.37@gslingacademy.com male False 2 True 19 Construction Engineer 99 9

SQL to find data with a partial match - in this example where the first character of the last name starts with Z

In [20]:
print("Data with a partial match for the last name (starting with Z):")
run_sql_command(cursor, """SELECT * FROM my_table
  WHERE last_name LIKE 'Z%'""")

Data with a partial match for the last name (starting with Z):
0:287 Penny Zuniga penny.zuniga.287@gslingacademy.com female False 7 True 30 Lawyer 84 97 60 100 83 88 94
1:484 Julie Zamora julie.zamora.484@gslingacademy.com female False 6 True 1 Unknown 76 82 93 98 98 60 65
2:1101 Sabrina Zavala sabrina.zavala.1101@gslingacademy.com female False 1 False 31 Scientist 92 78 96 91 82 98 100
3:1233 Toni Zimmerman toni.zimmerman.1233@gslingacademy.com female False 5 False 31 Accountant 94 98 74 94 35 95 80
4:1671 Charlotte Zuniga charlotte.zuniga.1671@gslingacademy.com female False 8 False 34 Lawyer 83 94 93 62 92 98 94


SQL to findthe average of a field (cast as an int as the data is stored as String here)

In [21]:
print("Average math score:")
run_sql_command(cursor, """SELECT AVG(CAST(math_score AS INTEGER))
  FROM my_table""")

Average math score:
0:83.452


SQL to find the maximum number in a field/ column

In [22]:
print("Maximum math score:")
run_sql_command(cursor, """SELECT MAX(CAST(math_score AS INTEGER))
  FROM my_table""")

Maximum math score:
0:100


SQL to find the minimum number in a field/ column

In [23]:
print("Minimum math score:")
run_sql_command(cursor, """SELECT MIN(CAST(math_score AS INTEGER))
  FROM my_table""")

Minimum math score:
0:40


SQL to find out how many students have chosen each career aspiration and display the results.

In [24]:
print("Group by career aspiration and count the rows:")
run_sql_command(cursor, """SELECT career_aspiration, COUNT(*) FROM my_table
  GROUP BY career_aspiration""")

Group by career aspiration and count the rows:
0:Accountant 126
1:Artist 67
2:Banker 169
3:Business Owner 309
4:Construction Engineer 68
5:Designer 56
6:Doctor 119
7:Game Developer 63
8:Government Officer 61
9:Lawyer 138
10:Real Estate Developer 83
11:Scientist 39
12:Software Engineer 315
13:Stock Investor 73
14:Teacher 59
15:Unknown 223
16:Writer 32


SQL to find all students with a part time job and a maths score > 0

In [25]:
print("Students with a part-time job and a math score greater than 90:")
run_sql_command(cursor, """SELECT * FROM my_table
  WHERE part_time_job = 'True' AND CAST(math_score AS INTEGER) > 90""")

Students with a part-time job and a math score greater than 90:
0:8 George Short george.short.8@gslingacademy.com male True 2 True 34 Software Engineer 95 95 82 63 84 70 85
1:13 Patricia Gomez patricia.gomez.13@gslingacademy.com female True 7 False 4 Business Owner 94 59 69 67 89 65 73
2:21 Tim Nichols tim.nichols.21@gslingacademy.com male True 3 False 15 Software Engineer 100 90 72 98 73 97 72
3:43 Lauren Farmer lauren.farmer.43@gslingacademy.com female True 6 False 3 Business Owner 93 94 92 77 87 98 63
4:58 David Vaughn david.vaughn.58@gslingacademy.com male True 2 False 18 Banker 94 78 73 97 67 78 99
5:99 Derrick Figueroa derrick.figueroa.99@gslingacademy.com male True 0 False 28 Lawyer 100 93 85 61 96 83 86
6:131 Amanda Vasquez amanda.vasquez.131@gslingacademy.com female True 5 False 25 Stock Investor 96 73 90 85 63 84 87
7:132 Chelsea Obrien chelsea.obrien.132@gslingacademy.com female True 4 False 16 Banker 95 82 66 63 67 92 69
8:158 Brian Sanchez brian.sanchez.158@gslingacademy.c

SQL to find the top 5 students with the highest maths scores

In [26]:
print("Top 5 students with the highest math scores:")
run_sql_command(cursor, """SELECT * FROM my_table
  ORDER BY CAST(math_score AS INTEGER)
  DESC LIMIT 5""")

Top 5 students with the highest math scores:
0:21 Tim Nichols tim.nichols.21@gslingacademy.com male True 3 False 15 Software Engineer 100 90 72 98 73 97 72
1:27 Jason Williams jason.williams.27@gslingacademy.com male False 3 False 34 Banker 100 77 80 94 63 90 90
2:50 Sonia Noble sonia.noble.50@gslingacademy.com female False 0 False 14 Accountant 100 89 90 93 30 83 74
3:96 Victoria Jones victoria.jones.96@gslingacademy.com female False 2 True 34 Software Engineer 100 98 89 85 81 64 94
4:99 Derrick Figueroa derrick.figueroa.99@gslingacademy.com male True 0 False 28 Lawyer 100 93 85 61 96 83 86


SQL to find all the career aspirations in the my_table that have more than 5 students aspiring to them. It then displays those career aspirations and the number of students who have selected each one.

In [27]:
print("Career aspirations with more than 5 students:")
run_sql_command(cursor, """SELECT career_aspiration, COUNT(*)
  FROM my_table
  GROUP BY career_aspiration
  HAVING COUNT(*) > 5""")

Career aspirations with more than 5 students:
0:Accountant 126
1:Artist 67
2:Banker 169
3:Business Owner 309
4:Construction Engineer 68
5:Designer 56
6:Doctor 119
7:Game Developer 63
8:Government Officer 61
9:Lawyer 138
10:Real Estate Developer 83
11:Scientist 39
12:Software Engineer 315
13:Stock Investor 73
14:Teacher 59
15:Unknown 223
16:Writer 32


SQL to find the total maths scores of the students inthe table

In [28]:
print("Total math score of all students:")
run_sql_command(cursor, """SELECT SUM(CAST(math_score AS INTEGER))
  FROM my_table""")

Total math score of all students:
0:166904
